In [ ]:
# ==========================================
# INSTALL LIBRARIES
# ==========================================

!pip install transformers sentence-transformers keybert nltk torch scikit-learn

In [35]:
import os
import re
import torch
import nltk

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from keybert import KeyBERT
from nltk.tokenize import sent_tokenize

nltk.download("punkt")
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ankit\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ankit\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [56]:
current_path = os.getcwd()
parent_path = os.path.dirname(current_path)
artifacts_path = current_path+"/artifacts/notes_model"
path_to_dataset_notes = parent_path+"/Dataset/AI_Notes"

In [57]:
artifacts_path

'E:\\Code\\PragyaPythonProject\\ImageWebsite\\AIModel\\Model/artifacts/notes_model'

In [38]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

Running on: cpu


In [39]:
MODEL_NAME = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

embedding_model = SentenceTransformer("all-mpnet-base-v2")

kw_model = KeyBERT(model=embedding_model)

print("Models loaded successfully")

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded successfully


In [40]:
def clean_text(text):

    # remove code blocks
    text = re.sub(r'```.*?```', '', text, flags=re.DOTALL)

    # remove java code lines
    text = re.sub(r'import .*?;', '', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [41]:
def split_text(text, max_tokens=900):

    sentences = sent_tokenize(text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:

        if len(tokenizer.encode(current_chunk + sentence)) < max_tokens:
            current_chunk += sentence + " "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + " "

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

In [42]:
def summarize_chunk(chunk):

    inputs = tokenizer(
        chunk,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=200,
        min_length=80,
        num_beams=8,
        length_penalty=1.2,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

In [43]:
def summarize_notes(text):

    text = clean_text(text)

    # split by headings or numbered sections
    sections = re.split(r'\n|\d+\.', text)

    summaries = []

    for section in sections:

        if len(section.split()) < 40:
            continue

        inputs = tokenizer(
            section,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=120,
            min_length=40,
            num_beams=6,
            no_repeat_ngram_size=3
        )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        summaries.append(summary)

    final_summary = " ".join(summaries)

    return final_summary

In [44]:
def bullet_summary(text):

    summary = summarize_notes(text)

    sentences = sent_tokenize(summary)

    bullets = []

    for s in sentences:

        bullets.append("• " + s)

    return "\n".join(bullets)

In [45]:
def extract_keywords(text):

    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1,3),
        stop_words="english",
        top_n=10
    )

    return [k[0] for k in keywords]

In [46]:
text = input("Enter your notes: ")

print("\nSUMMARY\n")
print(summarize_notes(text))

print("\nBULLET NOTES\n")
print(bullet_summary(text))

print("\nKEYWORDS\n")
print(extract_keywords(text))

Enter your notes:  Java memory management is the process by which the Java Virtual Machine (JVM) automatically handles the allocation and deallocation of memory. It uses a garbage collection to reclaim memory by removing unused objects, eliminating the need for manual memory management  JVM Memory Structure The JVM divides memory into several runtime data areas. Some areas are common for the entire JVM, while others are created separately for each thread.  jvm-memory JVM Language Classes 1. Heap Area Heap is a shared runtime data area where objects and arrays are stored. It is created when the JVM starts. JVM allows user to adjust the heap size. When the new keyword is used the object is allocated in the heap and its reference is stored in the stack. There exists one and only one heap for a running JVM process. Scanner sc = new Scanner(System.in)  Here, the Scanner object is stored in the heap and the reference sc is stored in the stack    Note: Garbage collection in heap area is manda


SUMMARY

Java memory management is the process by which the Java Virtual Machine (JVM) automatically handles the allocation and deallocation of memory. It uses a garbage collection to reclaim memory by removing unused objects, eliminating the need for manual memory management. Heap Area Heap is a shared runtime data area where objects and arrays are stored. It is created when the JVM starts. There exists one and only one heap for a running JVM process. Garbage collection in heap area is mandatory. Method area is used to store class-level information such as class structures, Method bytecode, Static variables, Constant pool, Interfaces. Method area can be of fixed or dynamic size depending on the system's configuration. Garbage collection of the method area is not guaranteed and depends on JVM implementation.

BULLET NOTES

• Java memory management is the process by which the Java Virtual Machine (JVM) automatically handles the allocation and deallocation of memory.
• It uses a garbage

In [58]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os

MODEL_NAME = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

os.makedirs(artifacts_path, exist_ok=True)

tokenizer.save_pretrained(artifacts_path)
model.save_pretrained(artifacts_path)

print("✅ Model saved at:", artifacts_path)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved at: E:\Code\PragyaPythonProject\ImageWebsite\AIModel\Model/artifacts/notes_model
